<a href="https://colab.research.google.com/github/nirvair09/advance_mango_detection_model/blob/main/advance_mango_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mango Detection and Ripeness Classification

This upgraded notebook builds a final-year-project-level mango analysis pipeline with three major goals:
1. Train a **YOLOv8 detector** to localize mangoes in images.
2. Train and compare **EfficientNetB0** and **MobileNetV2** classifiers for ripeness prediction.
3. Export the best detector-classifier pair for a **Streamlit deployment pipeline**.

Compared with the earlier version, this notebook adds:
- stronger training defaults,
- class-distribution analysis,
- train/validation/test splitting,
- callback-based training,
- learning-curve plots,
- confusion matrices and classification reports,
- backbone comparison between EfficientNetB0 and MobileNetV2,
- artifact export for deployment.


In [ ]:
# Install the core libraries used in this notebook.
# Run this once in a fresh Google Colab environment.
!pip install -q ultralytics roboflow tensorflow streamlit opencv-python kaggle seaborn scikit-learn pandas matplotlib

## 1. Import Libraries and Define Project Settings

Centralizing imports, paths, thresholds, and training constants makes the notebook easier to maintain and helps keep experiments reproducible.

In [ ]:
import json
import os
import random
import shutil
from pathlib import Path
from typing import Dict, List, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from IPython.display import Image, display
from roboflow import Roboflow
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import Model, layers
from tensorflow.keras.applications import EfficientNetB0, MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from ultralytics import YOLO

sns.set_theme(style="whitegrid")


def seed_everything(seed: int = 42) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


seed_everything(42)

# Roboflow configuration.
# Prefer storing secrets in environment variables when possible.
ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY", "Lkq2ojJa2S9mNjGNuviJ")
WORKSPACE_NAME = "pigrecipe"
PROJECT_NAME = "mango-7l7ug-napwv-razzf"
PROJECT_VERSION = 1

# Local project paths.
DATASET_DIR = Path("mango-1")
YOLO_DATA_CONFIG = DATASET_DIR / "data.yaml"
YOLO_BASE_WEIGHTS = "yolov8n.pt"
YOLO_RUNS_DIR = Path("/content/runs/detect")
YOLO_EXPERIMENT_NAME = "mango_final"
YOLO_TRAIN_DIR = YOLO_RUNS_DIR / YOLO_EXPERIMENT_NAME
YOLO_BEST_WEIGHTS = YOLO_TRAIN_DIR / "weights" / "best.pt"
PREDICT_DIR = YOLO_RUNS_DIR / "predict_final"
CROP_OUTPUT_DIR = Path("/content/mango_crops")
RIPENESS_DATA_DIR = Path("/content/ripeness_data")
CLASSIFIER_OUTPUT_DIR = Path("/content/classifier_artifacts")
EXPORT_DIR = Path("/content/exported_models")
CLASSIFIER_MODEL_PATH = EXPORT_DIR / "best_classifier.keras"
MODEL_METADATA_PATH = EXPORT_DIR / "model_metadata.json"

# Shared inference settings.
CLASS_NAMES = ["raw", "ripe", "overripe"]
CLASSIFIER_BACKBONES = ["efficientnetb0", "mobilenetv2"]
DETECTION_CONF_THRESH = 0.5
CLASSIFICATION_CONF_THRESH = 0.6
CROP_MARGIN = 10
CLASSIFIER_IMAGE_SIZE = (224, 224)
RANDOM_SEED = 42

# Training defaults selected for stronger project-level experiments.
YOLO_EPOCHS = 75
YOLO_BATCH_SIZE = 16
YOLO_IMAGE_SIZE = 640
YOLO_PATIENCE = 20
CLASSIFIER_BATCH_SIZE = 32
CLASSIFIER_EPOCHS = 25
FINE_TUNE_EPOCHS = 8
BASE_LEARNING_RATE = 1e-3
FINE_TUNE_LEARNING_RATE = 1e-5

CLASSIFIER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

## 2. Build Reusable Utilities

These helper functions support robust inference, structured dataset preparation, classifier construction, training visualization, and model evaluation.

In [ ]:
def detect_mangoes(model, img: np.ndarray, conf_thresh: float = DETECTION_CONF_THRESH) -> List[Tuple[int, int, int, int, float]]:
    """Return high-confidence mango boxes from a detector prediction."""
    results = model.predict(img, conf=conf_thresh, verbose=False)
    boxes = []

    for result in results:
        if result.boxes is None or len(result.boxes) == 0:
            continue

        xyxy = result.boxes.xyxy.cpu().numpy()
        confidences = result.boxes.conf.cpu().numpy()

        for box, conf in zip(xyxy, confidences):
            if float(conf) < conf_thresh:
                continue
            x1, y1, x2, y2 = map(int, box)
            boxes.append((x1, y1, x2, y2, float(conf)))

    return boxes


def crop_mango(img: np.ndarray, box: Tuple[int, int, int, int, float], margin: int = CROP_MARGIN) -> np.ndarray:
    """Crop a mango with a small safety margin to reduce tight-box errors."""
    x1, y1, x2, y2, _ = box
    height, width = img.shape[:2]

    x1 = max(0, x1 - margin)
    y1 = max(0, y1 - margin)
    x2 = min(width, x2 + margin)
    y2 = min(height, y2 + margin)

    return img[y1:y2, x1:x2]


def preprocess_crop(crop: np.ndarray) -> np.ndarray:
    """Resize and normalize a crop before classification."""
    resized = cv2.resize(crop, CLASSIFIER_IMAGE_SIZE).astype("float32") / 255.0
    return np.expand_dims(resized, axis=0)


def classify_mango(model, crop: np.ndarray, conf_thresh: float = CLASSIFICATION_CONF_THRESH) -> Tuple[str, float]:
    """Classify a crop and return an uncertain label when confidence is too low."""
    prediction = model.predict(preprocess_crop(crop), verbose=False)[0]
    confidence = float(np.max(prediction))
    label_idx = int(np.argmax(prediction))

    if confidence < conf_thresh:
        return "uncertain", confidence

    return CLASS_NAMES[label_idx], confidence


def process_image(
    img: np.ndarray,
    det_model,
    clf_model,
    det_conf_thresh: float = DETECTION_CONF_THRESH,
    clf_conf_thresh: float = CLASSIFICATION_CONF_THRESH,
    margin: int = CROP_MARGIN,
):
    """Detect mangoes, classify confident crops, and annotate the image."""
    annotated = img.copy()
    boxes = detect_mangoes(det_model, annotated, conf_thresh=det_conf_thresh)

    if len(boxes) == 0:
        return None, {
            "message": "No mango detected",
            "counts": {name: 0 for name in CLASS_NAMES},
            "avg_detection_conf": 0.0,
            "avg_combined_conf": 0.0,
            "results": [],
        }

    counts = {name: 0 for name in CLASS_NAMES}
    per_mango_results = []
    combined_scores = []

    for box in boxes:
        crop = crop_mango(annotated, box, margin=margin)
        if crop.size == 0:
            continue

        label, cls_conf = classify_mango(clf_model, crop, conf_thresh=clf_conf_thresh)
        x1, y1, x2, y2, det_conf = box
        combined_conf = float(det_conf * cls_conf)

        if label != "uncertain":
            counts[label] += 1

        per_mango_results.append({
            "label": label,
            "detection_conf": float(det_conf),
            "classification_conf": float(cls_conf),
            "combined_conf": combined_conf,
            "box": [x1, y1, x2, y2],
        })
        combined_scores.append(combined_conf)

        cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            annotated,
            f"{label} | det:{det_conf:.2f} cls:{cls_conf:.2f}",
            (x1, max(y1 - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (0, 255, 0),
            2,
        )

    summary = {
        "message": "ok",
        "counts": counts,
        "avg_detection_conf": float(np.mean([box[4] for box in boxes])) if boxes else 0.0,
        "avg_combined_conf": float(np.mean(combined_scores)) if combined_scores else 0.0,
        "results": per_mango_results,
    }
    return annotated, summary


def build_image_dataframe(root_dir: Path) -> pd.DataFrame:
    """Create a dataframe with image paths and class labels from class folders."""
    records = []
    valid_suffixes = {".jpg", ".jpeg", ".png", ".bmp"}

    for class_dir in sorted(root_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        for image_path in sorted(class_dir.rglob("*")):
            if image_path.suffix.lower() in valid_suffixes:
                records.append({"filepath": str(image_path), "label": class_dir.name})

    df = pd.DataFrame(records)
    if df.empty:
        raise ValueError(f"No class-organized images found in {root_dir}")

    return df


def plot_class_distribution(df: pd.DataFrame, title: str) -> None:
    plt.figure(figsize=(7, 4))
    ax = sns.countplot(data=df, x="label", order=sorted(df["label"].unique()), palette="viridis")
    plt.title(title)
    plt.xlabel("Class")
    plt.ylabel("Image count")
    for container in ax.containers:
        ax.bar_label(container, padding=2)
    plt.tight_layout()
    plt.show()


def make_data_splits(df: pd.DataFrame):
    train_df, temp_df = train_test_split(
        df,
        test_size=0.30,
        stratify=df["label"],
        random_state=RANDOM_SEED,
    )
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        stratify=temp_df["label"],
        random_state=RANDOM_SEED,
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


def build_generators(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame):
    train_datagen = ImageDataGenerator(
        rotation_range=20,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.15,
        shear_range=0.1,
        horizontal_flip=True,
        fill_mode="nearest",
        rescale=1.0 / 255.0,
    )
    eval_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

    train_data = train_datagen.flow_from_dataframe(
        train_df,
        x_col="filepath",
        y_col="label",
        target_size=CLASSIFIER_IMAGE_SIZE,
        batch_size=CLASSIFIER_BATCH_SIZE,
        class_mode="categorical",
        shuffle=True,
        seed=RANDOM_SEED,
    )
    val_data = eval_datagen.flow_from_dataframe(
        val_df,
        x_col="filepath",
        y_col="label",
        target_size=CLASSIFIER_IMAGE_SIZE,
        batch_size=CLASSIFIER_BATCH_SIZE,
        class_mode="categorical",
        shuffle=False,
    )
    test_data = eval_datagen.flow_from_dataframe(
        test_df,
        x_col="filepath",
        y_col="label",
        target_size=CLASSIFIER_IMAGE_SIZE,
        batch_size=CLASSIFIER_BATCH_SIZE,
        class_mode="categorical",
        shuffle=False,
    )
    return train_data, val_data, test_data


def build_classifier(backbone_name: str, num_classes: int) -> Model:
    backbone_name = backbone_name.lower()
    if backbone_name == "efficientnetb0":
        backbone = EfficientNetB0(include_top=False, weights="imagenet", input_shape=CLASSIFIER_IMAGE_SIZE + (3,))
    elif backbone_name == "mobilenetv2":
        backbone = MobileNetV2(include_top=False, weights="imagenet", input_shape=CLASSIFIER_IMAGE_SIZE + (3,))
    else:
        raise ValueError(f"Unsupported backbone: {backbone_name}")

    backbone._name = f"{backbone_name}_backbone"
    backbone.trainable = False

    inputs = layers.Input(shape=CLASSIFIER_IMAGE_SIZE + (3,))
    x = backbone(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.35)(x)
    x = layers.Dense(128, activation="relu")(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = Model(inputs, outputs, name=f"{backbone_name}_classifier")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(BASE_LEARNING_RATE),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def get_callbacks(backbone_name: str):
    checkpoint_path = CLASSIFIER_OUTPUT_DIR / f"{backbone_name}_best.keras"
    return [
        EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, min_lr=1e-6),
        ModelCheckpoint(filepath=str(checkpoint_path), monitor="val_accuracy", save_best_only=True),
    ]


def merge_histories(history_a, history_b):
    merged = {}
    for key in set(history_a.keys()).union(history_b.keys()):
        merged[key] = list(history_a.get(key, [])) + list(history_b.get(key, []))
    return merged


def plot_training_history(history_dict: Dict[str, List[float]], model_name: str) -> None:
    epochs = range(1, len(history_dict.get("accuracy", [])) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history_dict.get("accuracy", []), label="Train")
    axes[0].plot(epochs, history_dict.get("val_accuracy", []), label="Validation")
    axes[0].set_title(f"{model_name} Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()

    axes[1].plot(epochs, history_dict.get("loss", []), label="Train")
    axes[1].plot(epochs, history_dict.get("val_loss", []), label="Validation")
    axes[1].set_title(f"{model_name} Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


def evaluate_classifier_model(model, test_data, class_names: List[str], model_name: str) -> Dict[str, float]:
    predictions = model.predict(test_data, verbose=False)
    predicted_indices = np.argmax(predictions, axis=1)
    true_indices = test_data.classes
    ordered_class_names = [class_name for class_name, _ in sorted(test_data.class_indices.items(), key=lambda item: item[1])]

    accuracy = accuracy_score(true_indices, predicted_indices)
    weighted_f1 = f1_score(true_indices, predicted_indices, average="weighted")

    cm = confusion_matrix(true_indices, predicted_indices)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=ordered_class_names, yticklabels=ordered_class_names)
    plt.title(f"{model_name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

    print(f"Classification report for {model_name}:")
    print(classification_report(true_indices, predicted_indices, target_names=ordered_class_names, digits=4))

    return {
        "model": model_name,
        "test_accuracy": float(accuracy),
        "weighted_f1": float(weighted_f1),
    }

## 3. Download the Detection Dataset from Roboflow

This dataset is used to train YOLOv8 to localize mangoes in images.

In [ ]:
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE_NAME).project(PROJECT_NAME)
dataset = project.version(PROJECT_VERSION).download("yolov8")

print(f"Dataset downloaded to: {dataset.location}")
print("Contents:")
!ls mango-1

## 4. Train the YOLOv8 Mango Detector

The training schedule below is stronger than the earlier draft and is more suitable for a final-year-project-level experiment. The epoch count is increased to `75` with patience-based stopping support, while keeping `yolov8n` for a strong speed-quality trade-off.

In [ ]:
detector = YOLO(YOLO_BASE_WEIGHTS)

detector.train(
    data=str(YOLO_DATA_CONFIG),
    epochs=YOLO_EPOCHS,
    imgsz=YOLO_IMAGE_SIZE,
    batch=YOLO_BATCH_SIZE,
    patience=YOLO_PATIENCE,
    project=str(YOLO_RUNS_DIR),
    name=YOLO_EXPERIMENT_NAME,
    exist_ok=True,
    plots=True,
)

## 5. Review Detector Results and Visual Outputs

This section reloads the best YOLO weights, evaluates the detector on held-out data, and previews saved prediction samples and training plots.

In [ ]:
detector = YOLO(str(YOLO_BEST_WEIGHTS))

detection_metrics = detector.val(data=str(YOLO_DATA_CONFIG), split="test")
print(detection_metrics)

detector.predict(
    source=str(DATASET_DIR / "test" / "images"),
    save=True,
    conf=DETECTION_CONF_THRESH,
    project=str(YOLO_RUNS_DIR),
    name="predict_final",
    exist_ok=True,
)

In [ ]:
Image(filename=str(YOLO_TRAIN_DIR / "results.png"))

In [ ]:
prediction_images = sorted(os.listdir(PREDICT_DIR))
for image_name in prediction_images[:5]:
    display(Image(filename=str(PREDICT_DIR / image_name)))

## 6. Crop Detected Mangoes for Ripeness Classification

The detector output is reused to create mango crops that can later be passed through the classifier. This keeps the full pipeline consistent from training to deployment.

In [ ]:
CROP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

crop_count = 0
train_image_paths = sorted((DATASET_DIR / "train" / "images").glob("*"))

for image_path in train_image_paths:
    image_array = cv2.imread(str(image_path))
    if image_array is None:
        continue

    boxes = detect_mangoes(detector, image_array, conf_thresh=DETECTION_CONF_THRESH)

    for box in boxes:
        crop = crop_mango(image_array, box, margin=CROP_MARGIN)
        if crop.size == 0:
            continue

        crop_path = CROP_OUTPUT_DIR / f"mango_{crop_count}.jpg"
        cv2.imwrite(str(crop_path), crop)
        crop_count += 1

print(f"Saved {crop_count} mango crops to {CROP_OUTPUT_DIR}")

## 7. Download the Ripeness Dataset and Inspect Class Balance

A stronger classifier workflow starts with understanding the dataset distribution. This section downloads the ripeness dataset, creates a dataframe of image paths and labels, and visualizes class balance.

In [ ]:
!kaggle datasets download -d srabon00/mango-ripening-stage-classification
!unzip -q -o mango-ripening-stage-classification.zip -d /content/ripeness_data

ripeness_df = build_image_dataframe(RIPENESS_DATA_DIR)
print(ripeness_df.head())
print(ripeness_df["label"].value_counts())
plot_class_distribution(ripeness_df, "Ripeness Dataset Class Distribution")

## 8. Create Train / Validation / Test Splits

Instead of relying only on an internal validation split, this version creates explicit train, validation, and test partitions for better project-quality evaluation.

In [ ]:
train_df, val_df, test_df = make_data_splits(ripeness_df)

print(f"Train size: {len(train_df)}")
print(f"Validation size: {len(val_df)}")
print(f"Test size: {len(test_df)}")

plot_class_distribution(train_df, "Training Split Class Distribution")
plot_class_distribution(val_df, "Validation Split Class Distribution")
plot_class_distribution(test_df, "Test Split Class Distribution")

train_data, val_data, test_data = build_generators(train_df, val_df, test_df)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"],
)
class_weights = {
    train_data.class_indices[label]: float(weight)
    for label, weight in zip(np.unique(train_df["label"]), class_weights_array)
}

print("Class indices:", train_data.class_indices)
print("Class weights:", class_weights)

## 9. Train and Compare EfficientNetB0 and MobileNetV2

This section upgrades the classifier stage from a single-model experiment into a model-comparison study. Both backbones are trained with callbacks, and then lightly fine-tuned for stronger final performance.

In [ ]:
trained_models = {}
history_objects = {}
comparison_rows = []

for backbone_name in CLASSIFIER_BACKBONES:
    print(f"\nTraining backbone: {backbone_name}")
    model = build_classifier(backbone_name, num_classes=len(CLASS_NAMES))
    callbacks = get_callbacks(backbone_name)

    initial_history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=CLASSIFIER_EPOCHS,
        callbacks=callbacks,
        class_weight=class_weights,
        verbose=1,
    )

    model = tf.keras.models.load_model(CLASSIFIER_OUTPUT_DIR / f"{backbone_name}_best.keras")
    backbone_layer = model.get_layer(f"{backbone_name}_backbone")
    backbone_layer.trainable = True

    # Fine-tune only the last portion of the backbone to stabilize training.
    for layer in backbone_layer.layers[:-20]:
        layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(FINE_TUNE_LEARNING_RATE),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    fine_tune_history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=FINE_TUNE_EPOCHS,
        callbacks=callbacks,
        class_weight=class_weights,
        verbose=1,
    )

    combined_history = merge_histories(initial_history.history, fine_tune_history.history)
    history_objects[backbone_name] = combined_history
    trained_models[backbone_name] = model

    plot_training_history(combined_history, backbone_name.upper())
    metrics = evaluate_classifier_model(model, test_data, CLASS_NAMES, backbone_name.upper())
    comparison_rows.append(metrics)

comparison_df = pd.DataFrame(comparison_rows).sort_values(["weighted_f1", "test_accuracy"], ascending=False).reset_index(drop=True)
display(comparison_df)

best_classifier_name = comparison_df.loc[0, "model"].lower()
classifier = trained_models[best_classifier_name]
print(f"Selected best classifier: {best_classifier_name}")

In [ ]:
plt.figure(figsize=(7, 4))
comparison_plot_df = comparison_df.melt(id_vars="model", value_vars=["test_accuracy", "weighted_f1"], var_name="metric", value_name="score")
sns.barplot(data=comparison_plot_df, x="model", y="score", hue="metric", palette="Set2")
plt.title("Classifier Backbone Comparison")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## 10. Test the Full Detection + Classification Pipeline

This step uses the best classifier selected from the comparison study and applies the final end-to-end pipeline to a sample test image.

In [ ]:
sample_test_image_path = sorted((DATASET_DIR / "test" / "images").glob("*"))[0]
sample_bgr = cv2.imread(str(sample_test_image_path))
processed_image, summary = process_image(sample_bgr, detector, classifier)

plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(sample_bgr, cv2.COLOR_BGR2RGB))
plt.title("Original Test Image")
plt.axis("off")
plt.show()

if processed_image is None:
    print(summary["message"])
else:
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(processed_image, cv2.COLOR_BGR2RGB))
    plt.title("End-to-End Detection and Ripeness Prediction")
    plt.axis("off")
    plt.show()

    print("Counts:", summary["counts"])
    print(f"Average detection confidence: {summary['avg_detection_conf']:.2f}")
    print(f"Average combined confidence: {summary['avg_combined_conf']:.2f}")
    display(pd.DataFrame(summary["results"]))

In [ ]:
sample_crop_path = sorted(CROP_OUTPUT_DIR.glob("*.jpg"))[0]
sample_image = image.load_img(sample_crop_path, target_size=CLASSIFIER_IMAGE_SIZE)
sample_array = image.img_to_array(sample_image)
label, confidence = classify_mango(classifier, sample_array)

plt.figure(figsize=(4, 4))
plt.imshow(sample_image)
plt.title(f"Predicted: {label} | confidence: {confidence:.2f}")
plt.axis("off")
plt.show()

## 11. Export the Best Models and Experiment Metadata

The exported artifacts below can be used directly by the deployment app or reused for future experiments.

In [ ]:
shutil.copy2(YOLO_BEST_WEIGHTS, EXPORT_DIR / "best.pt")

for backbone_name, model in trained_models.items():
    model.save(EXPORT_DIR / f"{backbone_name}_classifier.keras")

classifier.save(CLASSIFIER_MODEL_PATH)
comparison_df.to_csv(EXPORT_DIR / "classifier_comparison.csv", index=False)

metadata = {
    "best_classifier": best_classifier_name,
    "class_names": CLASS_NAMES,
    "detection_conf_threshold": DETECTION_CONF_THRESH,
    "classification_conf_threshold": CLASSIFICATION_CONF_THRESH,
    "crop_margin": CROP_MARGIN,
    "classifier_image_size": CLASSIFIER_IMAGE_SIZE,
    "yolo_epochs": YOLO_EPOCHS,
    "classifier_epochs": CLASSIFIER_EPOCHS,
    "fine_tune_epochs": FINE_TUNE_EPOCHS,
}
MODEL_METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(f"Exported YOLO weights to: {EXPORT_DIR / 'best.pt'}")
print(f"Exported best classifier to: {CLASSIFIER_MODEL_PATH}")
print(f"Saved metadata to: {MODEL_METADATA_PATH}")
print(f"Saved comparison table to: {EXPORT_DIR / 'classifier_comparison.csv'}")

## 12. Final-Year Project Scope of Improvement

This notebook now includes stronger project-quality components, but the following upgrades would make it even stronger for publication or production-style deployment:

- add a larger and more diverse **negative detection set** so `No mango detected` is more reliable,
- introduce **k-fold validation** for the ripeness classifier,
- log experiments with **Weights & Biases or MLflow**,
- add **Grad-CAM** heatmaps for classifier interpretability,
- export the detector and classifier to **ONNX / TensorFlow Lite** for lightweight deployment,
- create a dedicated **test set of real-world market images** with mixed lighting and clutter,
- add **ablation studies** for crop margin, confidence thresholds, and augmentation settings.


## 13. Build a Streamlit App

The following cell writes a deployment-ready demo app that loads the exported best models and uses the same end-to-end logic as the notebook.

In [ ]:
%%writefile app.py
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import streamlit as st
from tensorflow.keras.models import load_model
from ultralytics import YOLO

EXPORT_DIR = Path("/content/exported_models")
DETECTION_MODEL_PATH = EXPORT_DIR / "best.pt"
CLASSIFICATION_MODEL_PATH = EXPORT_DIR / "best_classifier.keras"
METADATA_PATH = EXPORT_DIR / "model_metadata.json"

metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8")) if METADATA_PATH.exists() else {}
CLASS_NAMES = metadata.get("class_names", ["raw", "ripe", "overripe"])
DETECTION_CONF_THRESH = metadata.get("detection_conf_threshold", 0.5)
CLASSIFICATION_CONF_THRESH = metadata.get("classification_conf_threshold", 0.6)
CROP_MARGIN = metadata.get("crop_margin", 10)
IMAGE_SIZE = tuple(metadata.get("classifier_image_size", [224, 224]))
BEST_CLASSIFIER_NAME = metadata.get("best_classifier", "unknown")


@st.cache_resource
def load_models():
    detection_model = YOLO(str(DETECTION_MODEL_PATH))
    classification_model = load_model(CLASSIFICATION_MODEL_PATH)
    return detection_model, classification_model


def detect_mangoes(model, img, conf_thresh=DETECTION_CONF_THRESH):
    results = model.predict(img, conf=conf_thresh, verbose=False)
    boxes = []

    for result in results:
        if result.boxes is None or len(result.boxes) == 0:
            continue

        xyxy = result.boxes.xyxy.cpu().numpy()
        confidences = result.boxes.conf.cpu().numpy()

        for box, conf in zip(xyxy, confidences):
            if float(conf) < conf_thresh:
                continue
            x1, y1, x2, y2 = map(int, box)
            boxes.append((x1, y1, x2, y2, float(conf)))

    return boxes


def crop_mango(img, box, margin=CROP_MARGIN):
    x1, y1, x2, y2, _ = box
    height, width = img.shape[:2]
    x1 = max(0, x1 - margin)
    y1 = max(0, y1 - margin)
    x2 = min(width, x2 + margin)
    y2 = min(height, y2 + margin)
    return img[y1:y2, x1:x2]


def preprocess_crop(crop):
    resized = cv2.resize(crop, IMAGE_SIZE).astype("float32") / 255.0
    return np.expand_dims(resized, axis=0)


def classify_mango(model, crop, conf_thresh=CLASSIFICATION_CONF_THRESH):
    prediction = model.predict(preprocess_crop(crop), verbose=False)[0]
    confidence = float(np.max(prediction))
    label_idx = int(np.argmax(prediction))

    if confidence < conf_thresh:
        return "uncertain", confidence

    return CLASS_NAMES[label_idx], confidence


def process_image(img, det_model, clf_model, det_thresh, cls_thresh):
    annotated = img.copy()
    boxes = detect_mangoes(det_model, annotated, conf_thresh=det_thresh)

    if not boxes:
        return None, {
            "message": "No mango detected",
            "counts": {name: 0 for name in CLASS_NAMES},
            "results": [],
            "avg_detection_conf": 0.0,
            "avg_combined_conf": 0.0,
        }

    counts = {name: 0 for name in CLASS_NAMES}
    results_table = []
    combined_scores = []

    for index, box in enumerate(boxes, start=1):
        crop = crop_mango(annotated, box)
        if crop.size == 0:
            continue

        label, cls_conf = classify_mango(clf_model, crop, conf_thresh=cls_thresh)
        x1, y1, x2, y2, det_conf = box
        combined_conf = float(det_conf * cls_conf)
        combined_scores.append(combined_conf)

        if label != "uncertain":
            counts[label] += 1

        results_table.append({
            "id": index,
            "label": label,
            "detection_conf": round(float(det_conf), 4),
            "classification_conf": round(float(cls_conf), 4),
            "combined_conf": round(combined_conf, 4),
        })

        cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            annotated,
            f"{label} | det:{det_conf:.2f} cls:{cls_conf:.2f}",
            (x1, max(y1 - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (0, 255, 0),
            2,
        )

    return annotated, {
        "message": "ok",
        "counts": counts,
        "results": results_table,
        "avg_detection_conf": float(np.mean([box[4] for box in boxes])) if boxes else 0.0,
        "avg_combined_conf": float(np.mean(combined_scores)) if combined_scores else 0.0,
    }


def main():
    st.set_page_config(page_title="Mango Detection and Ripeness App", layout="centered")
    st.title("Mango Detection and Ripeness App")
    st.caption(f"Best classifier selected during training: {BEST_CLASSIFIER_NAME}")
    st.write("Upload an image to detect mangoes and classify ripeness for each detected fruit.")

    detection_model, classification_model = load_models()

    det_thresh = st.slider("Detection confidence threshold", 0.1, 0.95, float(DETECTION_CONF_THRESH), 0.05)
    cls_thresh = st.slider("Classification confidence threshold", 0.1, 0.95, float(CLASSIFICATION_CONF_THRESH), 0.05)
    uploaded_file = st.file_uploader("Upload an image", type=["jpg", "jpeg", "png"])

    if uploaded_file is None:
        return

    file_bytes = np.asarray(bytearray(uploaded_file.read()), dtype=np.uint8)
    image_array = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    st.image(image_array, caption="Uploaded image", channels="BGR", use_container_width=True)

    processed_image, summary = process_image(image_array, detection_model, classification_model, det_thresh, cls_thresh)

    if processed_image is None:
        st.error(summary["message"])
        return

    st.image(processed_image, caption="Detection and ripeness result", channels="BGR", use_container_width=True)
    st.write("Ripeness counts:", summary["counts"])
    st.write(f"Average detection confidence: {summary['avg_detection_conf']:.2f}")
    st.write(f"Average combined confidence: {summary['avg_combined_conf']:.2f}")
    st.dataframe(pd.DataFrame(summary["results"]), use_container_width=True)


if __name__ == "__main__":
    main()